In [ ]:
import pandas as pd
from tensorboard.backend.event_processing import event_accumulator
import os, re, shutil
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import gc

In [ ]:
data_path = '../train_outputs'

In [ ]:
def get_thr(ss: str):
    match = re.search(r'\(([\d\.]+)\)', ss)
    if match != None:
        return float(match.group(1))

def get_metric(ss: str):
    match = re.search(r'\([\d\.]+\)_(.*)$', ss)
    if match != None:
        return match.group(1)

def get_audio_type(ss: str):
    match = re.search(r'(std|pos|silence|sil|noise)', ss)
    if match != None:
        return match.group(1)

def get_dataset(ss: str):
    match = re.search(r'(avs_ms3|avs_s4|ms3|s4|vggss|exvggss|vggsound|flickr|exflickr|avatar_one_bb|avatar_one_seg|avatar_one)', ss)
    if match != None:
        return match.group(1)

def get_epoch(ss: str):
    match = re.search(r'epoch(\d+|best)', ss)
    if match != None:
        epoch = match.group(1)
        if epoch == 'best':
            epoch = 20
        return int(epoch)

def get_snr(ss: str):
    match = re.search(r'snr(\d+)', ss)
    if match == None:
        return -1
    snr = match.group(1)
    return int(snr)

def get_min_max(ss: str):
    match = re.search(r'(min|max)', ss)
    if match == None:
        return ''
    snr = match.group(1)
    return snr

def load_nested_tb_logs(root_dir):
    all_data = []

    # Walk through the directory tree
    for root, dirs, files in os.walk(root_dir):
        # Check if there are any tfevents files in this specific folder
        if any(f.startswith("events.out.tfevents") for f in files):
            # Extract the folder name to use as a category/run label

            # Initialize accumulator for this specific subdirectory
            acc = event_accumulator.EventAccumulator(root)
            acc.Reload()

            for tag in acc.Tags()['scalars']:
                events = acc.Scalars(tag)
                df_temp = pd.DataFrame(events)

                # We add 'metric' (e.g., value) and 'sub_dir' (e.g., test_noise_avs...)
                df_temp['metric_tag'] = tag
                df_temp['run_group'] = root

                all_data.append(df_temp)

    if not all_data:
        print("No event files found in the specified path.")
        return pd.DataFrame()

    # Combine all found data
    master_df = pd.concat(all_data, ignore_index=True)

    # Cleanup: Convert time and reorder columns
    master_df['wall_time'] = pd.to_datetime(master_df['wall_time'], unit='s')

    return master_df

def load_eval(path, run_name):
    df = load_nested_tb_logs(path)
    print(f"Loaded {len(df)} data points.")

    df['threshold'] = df['run_group'].apply(lambda x: get_thr(str(x)))
    df['metric'] = df['run_group'].apply(lambda x: get_metric(str(x)))
    df['audio_type'] = df['metric_tag'].apply(lambda x: get_audio_type(str(x)))
    df['dataset'] = df['run_group'].apply(lambda x: get_dataset(str(x)))
    df['epoch'] = df['run_group'].apply(lambda x: get_epoch(str(x)))
    df['snr'] = df['run_group'].apply(lambda x: get_snr(str(x)))
    df.drop(['wall_time', 'metric_tag', 'run_group'],axis=1, inplace=True)
    df = df.assign(run=run_name)

    return df

def load_nested_np_logs(root_dir):
    tmp_list = []
    for root, dirs, files in os.walk(root_dir):
        for f in files:
            if '.pkl' in f:
                # arr = np.load(os.path.join(root, f), allow_pickle=True)
                tmp_list.append([root, f])

    return pd.DataFrame(tmp_list, columns=['run_group', 'metric_tag'])

def load_infer_info(path, run_name):
    df = load_nested_np_logs(path)
    print(f"Loaded {len(df)} data points.")


    df['epoch'] = df['run_group'].apply(lambda x: get_epoch(str(x)))
    df['audio_type'] = df['metric_tag'].apply(lambda x: get_audio_type(str(x)))
    df['snr'] = df['run_group'].apply(lambda x: get_snr(str(x)))
    df['dataset'] = df['metric_tag'].apply(lambda x: get_dataset(str(x)))
    df['min_max'] = df['metric_tag'].apply(lambda x: get_min_max(str(x)))
    # df.drop(['metric_tag', 'run_group'], axis=1, inplace=True)
    df = df.assign(run=run_name)

    return df

In [ ]:
def print_metrics(df, epoch='all', thr=0.5):
    filtered_df = df[
        (df['threshold'] == thr) &
        (df['metric'].isin(['cIoU_hat', 'AUC', 'pIA_hat', 'AUC_N', 'mIoU', 'Fmeasure']))
    ]

    if epoch != 'all':
        filtered_df = filtered_df[filtered_df['epoch'] == epoch]

    # 2. Pivot the data
    # index: what you want as rows
    # columns: what you want as side-by-side columns
    # values: the numbers to fill the table
    pivot_df = filtered_df.pivot_table(
        index=['dataset', 'epoch'],
        columns=['audio_type', 'metric'],
        values='value',
    )

    # Define the desired order for each audio_type
    # std_cols = [('std', m) for m in ['cIoU_hat', 'AUC', 'mIoU', 'Fmeasure']]
    std_cols = [('std', m) for m in ['cIoU_hat', 'AUC']]
    silence_cols = [('silence', m) for m in ['pIA_hat', 'AUC_N']]
    noise_cols = [('noise', m) for m in ['pIA_hat', 'AUC_N']]

    # Combine them into one ordered list
    target_columns = std_cols + silence_cols + noise_cols

    # Reindex the columns to the new order
    # errors='ignore' ensures it doesn't crash if a specific metric is missing for one type
    pivot_df = pivot_df.reindex(columns=target_columns)

    pd.options.display.float_format = "{:,.3f}".format
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.options.display.width = 1000 # Increased width to prevent wrapping

    print(pivot_df)
    return pivot_df

def print_metrics_noisy(df, epoch='all', thr=0.5):
    filtered_df = df[
        (df['threshold'] == thr) &
        (df['metric'].isin(['cIoU_hat', 'AUC', 'pIA_hat', 'AUC_N', 'mIoU', 'Fmeasure']))
    ]

    if epoch != 'all':
        filtered_df = filtered_df[filtered_df['epoch'] == epoch]

    # 2. Pivot the data
    # index: what you want as rows
    # columns: what you want as side-by-side columns
    # values: the numbers to fill the table
    pivot_df = filtered_df.pivot_table(
        index=['dataset', 'epoch'],
        columns=['audio_type', 'snr', 'metric'],
        values='value',
    )

    # Define the desired order for each audio_type
    # std_cols = [('std', m) for m in ['cIoU_hat', 'AUC', 'mIoU', 'Fmeasure']]
    std_cols = [('std', s, m) for s in [5.0, 10.0, 20.0, -1] for m in ['cIoU_hat', 'AUC']]
    silence_cols = [('silence', -1, m) for m in ['pIA_hat', 'AUC_N']]
    noise_cols = [('noise', -1, m) for m in ['pIA_hat', 'AUC_N']]

    # Combine them into one ordered list
    target_columns = std_cols + silence_cols + noise_cols

    # Reindex the columns to the new order
    # errors='ignore' ensures it doesn't crash if a specific metric is missing for one type
    pivot_df = pivot_df.reindex(columns=target_columns)

    pd.options.display.float_format = "{:,.3f}".format
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.options.display.width = 1000 # Increased width to prevent wrapping

    print(pivot_df)
    return pivot_df

def print_runs(df, thr=0.5):
    filtered_df = df[
        (df['threshold'] == thr) &
        (df['metric'].isin(['cIoU_hat', 'AUC', 'pIA_hat', 'AUC_N', 'mIoU', 'Fmeasure']))
    ]

    pivot_df = filtered_df.pivot_table(
        index=['dataset', 'run'],
        columns=['audio_type', 'metric'],
        values='value',
    )

    std_cols = [('std', m) for m in ['cIoU_hat', 'AUC']]
    silence_cols = [('silence', m) for m in ['pIA_hat', 'AUC_N']]
    noise_cols = [('noise', m) for m in ['pIA_hat', 'AUC_N']]

    target_columns = std_cols + silence_cols + noise_cols

    pivot_df = pivot_df.reindex(columns=target_columns)

    pd.options.display.float_format = "{:,.3f}".format
    pd.options.display.max_columns = None
    pd.options.display.max_rows = None
    pd.options.display.width = 1000 # Increased width to prevent wrapping

    print(pivot_df)
    return pivot_df

def plot_all_metrics(df):
    color_palette = {}
    for dataset, color in zip(sorted(df['dataset'].unique()), sns.color_palette(n_colors=df['dataset'].nunique()).as_hex()):
        color_palette[dataset] = color
    # 1. Setup the style
    sns.set_theme(style="whitegrid")

    # 2. Define strict mappings
    # Mapping line styles to audio types
    style_map = {
        'std': (None, None),  # Solid
        'noise': (5, 5),      # Dashed
        'silence': (1, 2)     # Dotted
    }

    # 3. Get list of unique metrics
    # metrics = df['metric'].unique()
    metrics = ['AUC', 'AUC_N']

    for m in metrics:
        subset = df[df['metric'] == m].copy()
        subset = subset.sort_values('threshold')

        plt.figure(figsize=(10, 6))

        # 4. Create the lineplot with the fixed palette
        ax = sns.lineplot(
            data=subset,
            x='threshold',
            y='value',
            hue='dataset',
            palette=color_palette,  # Force consistent colors
            style='audio_type',
            dashes=style_map,
            markers=True,
            linewidth=2
        )

        # 5. Formatting
        plt.title(f"Metric: {m}", fontsize=15, fontweight='bold')
        plt.xlabel("Threshold")
        plt.ylabel("Value")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.xticks(np.arange(0, 1.1, 0.1))
        plt.yticks(np.arange(0, 1.1, 0.1))
        plt.xlim(0, 1)
        plt.ylim(0, 1)
        precision = 1
        # plt.ylim(np.true_divide(np.floor(subset['value'].min() * 10**precision), 10**precision),
        # np.true_divide(np.ceil(subset['value'].max() * 10**precision), 10**precision))


        plt.tight_layout()

        plt.show()

def plot_all_experiments(experiments):
    merged_df = pd.concat([e.metrics[e.metrics['epoch'] == e.best_epoch] for e in experiments.values()])

    merged_df = merged_df[
        (merged_df['threshold'] == 0.5) &
        (merged_df['dataset'] == 'avatar_one_seg')
    ]

    df = merged_df[merged_df['run'] != 'baseline']

    color_palette = {}
    for dataset, color in zip(sorted(df['run'].unique()), sns.color_palette(n_colors=df['run'].nunique()).as_hex()):
        color_palette[dataset] = color

    # 1. Setup the style
    sns.set_theme(style="whitegrid")

    # 2. Define strict mappings
    # Mapping line styles to audio types
    style_map = {
        'std': (None, None),  # Solid
        'noise': (5, 5),      # Dashed
        'silence': (1, 2)     # Dotted
    }

    df = df.sort_values('threshold')

    # 3. Get list of unique metrics
    # metrics = df['metric'].unique()
    metrics = ['AUC', 'AUC_N']

    for m in metrics:
        subset = df[df['metric'] == m].copy()
        subset = subset.sort_values('threshold')

        baseline_value = merged_df[
            (merged_df['run'] == 'baseline') &
            (merged_df['metric'] == m)
        ]['value'].to_list()[0]

        plt.figure(figsize=(10, 6))

        plt.hlines(baseline_value, 0, 20, label='baseline', linestyles='dashed')

        # 4. Create the lineplot with the fixed palette
        ax = sns.lineplot(
            data=subset,
            x='epoch',
            y='value',
            hue='run',
            palette=color_palette,  # Force consistent colors
            style='audio_type',
            dashes=style_map,
            markers=True,
            linewidth=2
        )

        # 5. Formatting
        plt.title(f"Evaluation @(threshold=0.5, avatar_seg, {m})", fontsize=15, fontweight='bold')
        plt.xlabel("Epoch")
        plt.ylabel("Value")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.yticks(np.arange(0, 1.1, 0.1))
        plt.xticks(range(0, 21, 2))
        precision = 1
        plt.ylim(
            np.true_divide(np.floor(min(subset['value'].min(), baseline_value) * 10**precision), 10**precision),
            np.true_divide(np.ceil(max(subset['value'].max(), baseline_value) * 10**precision), 10**precision)
        )

        plt.tight_layout()

        plt.show()

def get_thresholds(infer_info_df):

    df = infer_info_df[(infer_info_df["dataset"] == 'vggss') &
                       (infer_info_df["min_max"] == 'max')].copy()

    return_thresholds = {}

    for epoch in df['epoch'].unique():
        df_epoch = df[df['epoch'] == epoch].copy()

        df_epoch['data'] = df_epoch.apply(
            lambda x: np.load(os.path.join(x['run_group'], x['metric_tag']), allow_pickle=True).ravel(),
            axis=1
        )

        pos_arr = df_epoch[df_epoch['audio_type'] == 'pos']['data'].iloc[0]
        sil_arr = df_epoch[df_epoch['audio_type'] == 'sil']['data'].iloc[0]
        noise_arr = df_epoch[df_epoch['audio_type'] == 'noise']['data'].iloc[0]
        max_negatives = [sil_arr, noise_arr]
        max_negatives_separate = [np.percentile(sil_arr, 75), np.percentile(noise_arr, 75)]

        return_thresholds[int(epoch)] = {
            'max_neg': np.mean(max_negatives),
            'max_neg_plus_10': np.mean(max_negatives) * 1.1,
            'max_q2_pos': np.percentile(pos_arr, 25),
            'max_q3_all': np.percentile(max_negatives, 75),
            'max_q3_separate': np.max(max_negatives_separate)
        }

        del df_epoch, max_negatives, max_negatives_separate
        gc.collect()

    return return_thresholds

def boxplots_by_dataset(infer_info_df, dataset_name, threshold_dict, epochs, th_name = 'max_neg', min_max = 'max'):
    try:
        df = infer_info_df[(infer_info_df["dataset"] == dataset_name) &
                                        (infer_info_df["min_max"] == min_max)].copy()

        df = df[[e in epochs for e in df['epoch']]]
        col_order = sorted(df["epoch"].dropna().unique())

        df['data'] = df.apply(
            lambda x: np.load(os.path.join(x['run_group'], x['metric_tag']), allow_pickle=True).ravel(),
            axis=1
        )

        df = df.explode("data")

        fig = sns.catplot(df,
            y='data',
            hue='audio_type',
            hue_order=["pos", "sil", "noise"],
            kind="box",
            palette='pastel',
            col='epoch',
            col_order=col_order,
            height=4,
            aspect=0.5,
        )

        for ax, epoch in zip(fig.axes.flat, col_order):
            th_value = threshold_dict.get(int(epoch), {}).get(th_name)
            print(int(epoch), th_name)
            print(threshold_dict.get(int(epoch), {}))
            print(th_value)
            if th_value is not None:
                ax.axhline(float(th_value), color='crimson', linestyle='--', linewidth=1.5)

        plt.suptitle(f'{dataset_name} test set')
        sns.move_legend(fig, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3, title=None, frameon=False)
        fig.set_titles(y=-0.1)

        plt.tight_layout()
        fig.savefig("out.png")
        plt.show()

    finally:
        # Drop epoch data after plotting
        del df
        gc.collect()

In [ ]:
if "EXPERIMENTS" not in globals():
    EXPERIMENTS = {}

def multiton(cls):
    def getinstance(name):
        if name not in EXPERIMENTS:
            EXPERIMENTS[name] = cls(name)
        return EXPERIMENTS[name]
    return getinstance

In [ ]:
# EXPERIMENTS = {}

In [ ]:
@multiton
class Experiment(object):
    def __init__(self, exp_name) -> None:
        self.name = exp_name
        self.metrics = pd.DataFrame()
        self.infer_info = pd.DataFrame()
        self.loaded_dirs = []
        self.focus_epoch = -1
        self.focus_threshold = 0.5

    def load_eval_metrics(self, path_to_tensorboard_dir):
        if path_to_tensorboard_dir not in self.loaded_dirs:
            metrics = load_eval(path_to_tensorboard_dir, self.name)
            self.metrics = pd.concat([
                self.metrics,
                metrics
            ])
            self.loaded_dirs.append(path_to_tensorboard_dir)

    def load_eval_inference_info(self, path_to_numpy_dir):
        if path_to_numpy_dir not in self.loaded_dirs:
            self.infer_info = load_infer_info(path_to_numpy_dir, self.name)
            self.thresholds = get_thresholds(self.infer_info)
            print(self.thresholds)
            self.loaded_dirs.append(path_to_numpy_dir)

    def print_metrics(self, epoch=None, thr=None):
        if epoch == None:
            epoch = self.focus_epoch
        if thr == None:
            thr = self.focus_threshold
        print_metrics(self.metrics, epoch=epoch, thr=thr)

    def print_metrics_noisy(self, epoch=None, thr=None):
        if epoch == None:
            epoch = self.focus_epoch
        if thr == None:
            thr = self.focus_threshold
        print_metrics_noisy(self.metrics, epoch=epoch, thr=thr)

    def plot_all_metrics(self, epoch=None):
        if epoch == None:
            epoch = self.focus_epoch
        plot_all_metrics(self.metrics[self.metrics['epoch'] == epoch])

    def boxplots_by_dataset(self, dataset_name, epochs=None, th_name='max_neg', min_max='max'):
        if epochs == None:
            epochs = [self.focus_epoch]
        boxplots_by_dataset(self.infer_info, dataset_name, self.thresholds, epochs, th_name=th_name, min_max=min_max)

In [ ]:
exp_v2_B16 = Experiment('exp_v2_B16')
exp_v2_B16.load_eval_metrics(os.path.join(data_path, "2073339/Test_record/ACL_ViT16_aclifa_2gpu/tensorboard")) # unique-waterfall-33 or olive-sun-36
exp_v2_B16.load_eval_metrics(os.path.join(data_path, "2073339/Test_record_noisy/ACL_ViT16_Exp_ACL_v2/tensorboard"))
exp_v2_B16.load_eval_inference_info(os.path.join(data_path, "2073339/Test_record/ACL_ViT16_aclifa_2gpu/numpy"))

In [ ]:
exp_v2_B16.boxplots_by_dataset('avatar_one', [9, 10, 11, 12, 13], th_name='max_q2_pos')

In [ ]:
exp_v2_B16.focus_epoch = 19

In [ ]:
# _ = print_metrics_noisy(exp_v2_B16[exp_v2_B16['epoch'] == 19])
exp_v2_B16.print_metrics(epoch='all')

In [ ]:
exp_v2_B16.print_metrics_noisy()

In [ ]:
exp_v2_B16.plot_all_metrics()

In [ ]:
# plot_all_experiments(EXPERIMENTS)